In [1]:
import pandas as pd
import sys
import os
from pathlib import Path

project_root = Path.cwd().parent.parent
project_root_str = str(project_root)

if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

os.chdir(project_root_str)

from src.utils.fetch_player_stats import *
from src.utils.fetch_team_stats import *
from src.utils.helper_functions import *

## Fetches Player Gamelogs

In [2]:
nba = FetchPlayersStats()
data = nba.getCompleteStats(
    season='2025-26', 
    season_type='Regular Season', 
    sleep_time=2, 
    max_workers=5,
    batch_limit=100,
    complete_cache_file='data/raw/seaon_stats/S26.csv',
    include_playbyplay=False
)
data.tail()

Processing 7 games (limited by batch_limit=100)

Processing batch 1/1 (7 games)
Completed batch 1/1

Merging team stats...
Cache updated. Total games now: 354


,PLAYER_NAME,PLAYER_ID,MATCHUP,TEAM_ABBREVIATION,TEAM_ID,OPP_ABBREVIATION,HOME_GAME,GAME_ID,GAME_DATE,WL,...,OPP_OFF_RATING,OPP_PTS,OPP_FGM,OPP_FGA,OPP_FG_PCT,OPP_REB,OPP_AST,OPP_STL,OPP_BLK,OPP_TOV
7769,Chris Youngblood,1642959,OKC @ UTA,OKC,1610612760,UTA,0,0022500363,2025-12-07,W,...,99.7,101.0,40.0,85.0,0.471,37.0,25.0,6.0,3.0,17.0
7770,Tyrese Maxey,1630178,PHI vs. LAL,PHI,1610612755,LAL,1,0022500362,2025-12-07,L,...,113.3,112.0,41.0,85.0,0.482,52.0,25.0,3.0,8.0,10.0
7771,Branden Carlson,1642382,OKC @ UTA,OKC,1610612760,UTA,0,0022500363,2025-12-07,W,...,99.7,101.0,40.0,85.0,0.471,37.0,25.0,6.0,3.0,17.0
7772,Deandre Ayton,1629028,LAL @ PHI,LAL,1610612747,PHI,0,0022500362,2025-12-07,W,...,105.4,108.0,36.0,102.0,0.353,54.0,21.0,5.0,4.0,6.0
7773,Gabe Vincent,1629216,LAL @ PHI,LAL,1610612747,PHI,0,0022500362,2025-12-07,W,...,105.4,108.0,36.0,102.0,0.353,54.0,21.0,5.0,4.0,6.0


In [3]:
pd.set_option('display.max_columns', None)

# s19_regular = pd.read_csv('data/raw/seaon_stats/S19.csv')
# s20_regular = pd.read_csv('data/raw/seaon_stats/S20.csv')
# s21_regular = pd.read_csv('data/raw/seaon_stats/S21.csv')
s22_regular = pd.read_csv('data/raw/seaon_stats/S22.csv')
s23_regular = pd.read_csv('data/raw/seaon_stats/S23.csv')
s24_regular = pd.read_csv('data/raw/seaon_stats/S24.csv')
s25_regular = pd.read_csv('data/raw/seaon_stats/S25.csv')
s26_regular = pd.read_csv('data/raw/seaon_stats/S26.csv')

for df in [s22_regular, s23_regular, s24_regular, s25_regular, s26_regular]:
    df.drop(columns=['Unnamed: 0', 'DEF_FG_PCT_ALLOWED', 'DEF_3PT_PCT_ALLOWED', 'PTS_ALLOWED_PER_MIN', 
                     'DEF_TOV_FORCED_PER_MIN', 'DEF_BLOCKS_PER_MIN', 'DEF_SHOOTING_FOULS_PER_MIN', 
                     'DEF_AST_ALLOWED_PER_MIN'], errors='ignore', inplace=True)

## Fetching play by play data for model v2

In [4]:
# nba = FetchPlayersStats()
# data = nba.getCompleteStats(
#     season='2023-24', 
#     season_type='Regular Season', 
#     sleep_time=2, 
#     max_workers=5,
#     batch_limit=50,
#     complete_cache_file='../DATA/CSV_FILES/REGULAR_DATA/S24v2.csv',
#     include_playbyplay=True
# )
# data.tail()

## Assign features for regular season data

In [4]:
# data = [s22_regular,s23_regular,s24_regular, s25_regular, s26_regular]
# seasons = [2022,2023,2024, 2025, 2026]
data = [s26_regular]
seasons = [2026]

output_dir = os.path.join(project_root, 'data', 'processed', 'training')
os.makedirs(output_dir, exist_ok=True)

for season_data, year in zip(data, seasons):
    print(f"Processing year {year}...")
    
    # Process features
    processed_data = process_season_features(
        season_data, 
        prop_type='PTS',
        year=year
    )
    
    # Save file
    output_path = os.path.join(output_dir, f'PTS_TRAIN_{str(year)[-2:]}.csv')
    processed_data.to_csv(output_path)
    print(f"Completed {year}")

Processing year 2026...
Loading position cache...
Loaded 1245 players from cache
Found 497 unique players, 0 need to be fetched
Saving updated cache...
Cache saved with 1245 players
Applying positions to dataset...
Creating position flags...
Position assignment completed!
Completed 2026
